# RAG Pipeline — Complete Notebook
**Author:** minahil  
Builds a full Retrieval-Augmented Generation pipeline:
1. Data Ingestion (TXT + PDF)
2. Text Splitting / Chunking
3. Embeddings
4. Vector Store (ChromaDB)
5. RAG Retriever
6. End-to-End QA (prompt builder)
7. PDF Ingestion (your own PDFs + demo PDF)
8. HuggingFace LLM Integration (Mistral / Zephyr — free)

---
## 1. Data Ingestion

In [22]:
print("hello")

hello


In [23]:
## Data Ingestion

from langchain_core.documents import Document

In [24]:
doc = Document(
    page_content="this is the main page text context i am using this to create rag",
    metadata={
        "source": "example.txt",
        "page": 1,
        "author": "minahil",
        "date_created": "30-4-2025"
    }
)


In [25]:
doc

Document(metadata={'source': 'example.txt', 'page': 1, 'author': 'minahil', 'date_created': '30-4-2025'}, page_content='this is the main page text context i am using this to create rag')

In [26]:
import os

# Create folder
os.makedirs("data/text_files", exist_ok=True)
os.makedirs("data/pdf", exist_ok=True)

In [27]:
sample_text = {
    "data/text_files/AI_intro.txt": """
Artificial Intelligence (AI) is transforming industries by enabling machines to perform tasks that typically require human intelligence.

Machine Learning (ML), a subset of AI, allows systems to learn patterns from data and improve over time without being explicitly programmed.

Natural Language Processing (NLP) focuses on helping computers understand and generate human language.

Retrieval-Augmented Generation (RAG) combines retrieval with large language models for better responses.
""",

    "data/text_files/python_intro.txt": """
Python is a powerful and beginner-friendly programming language.

It is widely used in web development, data science, machine learning, automation, and artificial intelligence.

Python has simple syntax, making it easy to learn and use.

Popular Python libraries include NumPy, Pandas, TensorFlow, and Flask.
"""
}

for filepath, content in sample_text.items():
    with open(filepath, "w", encoding="utf-8") as f:
        f.write(content.strip())

print("2 sample text files created successfully")

2 sample text files created successfully


In [28]:
from langchain_community.document_loaders import TextLoader

loader = TextLoader("data/text_files/python_intro.txt", encoding="utf-8")

documents = loader.load()

documents

[Document(metadata={'source': 'data/text_files/python_intro.txt'}, page_content='Python is a powerful and beginner-friendly programming language.\n\nIt is widely used in web development, data science, machine learning, automation, and artificial intelligence.\n\nPython has simple syntax, making it easy to learn and use.\n\nPopular Python libraries include NumPy, Pandas, TensorFlow, and Flask.')]

In [29]:
from langchain_community.document_loaders import DirectoryLoader, TextLoader

dir_text = DirectoryLoader(
    "data/text_files/",
    glob="**/*.txt",
    loader_cls=TextLoader,
    loader_kwargs={"encoding": "utf-8"},
    show_progress=False
)

documents = dir_text.load()

print(f"Loaded {len(documents)} documents")
print(documents)

Loaded 2 documents
[Document(metadata={'source': 'data/text_files/python_intro.txt'}, page_content='Python is a powerful and beginner-friendly programming language.\n\nIt is widely used in web development, data science, machine learning, automation, and artificial intelligence.\n\nPython has simple syntax, making it easy to learn and use.\n\nPopular Python libraries include NumPy, Pandas, TensorFlow, and Flask.'), Document(metadata={'source': 'data/text_files/AI_intro.txt'}, page_content='Artificial Intelligence (AI) is transforming industries by enabling machines to perform tasks that typically require human intelligence.\n\nMachine Learning (ML), a subset of AI, allows systems to learn patterns from data and improve over time without being explicitly programmed.\n\nNatural Language Processing (NLP) focuses on helping computers understand and generate human language.\n\nRetrieval-Augmented Generation (RAG) combines retrieval with large language models for better responses.')]


---
## 2. Text Splitting / Chunking

Large documents need to be split into smaller, overlapping chunks before embedding.  
We use `RecursiveCharacterTextSplitter` which splits on `\n\n`, `\n`, then spaces — preserving semantic boundaries as much as possible.

In [30]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=200,       # max characters per chunk
    chunk_overlap=40,     # overlap between consecutive chunks to preserve context
    separators=["\n\n", "\n", " ", ""]
)

chunks = text_splitter.split_documents(documents)

print(f"Total documents : {len(documents)}")
print(f"Total chunks    : {len(chunks)}")
print()

for i, chunk in enumerate(chunks):
    print(f"--- Chunk {i+1} (source: {chunk.metadata.get('source', 'unknown')}) ---")
    print(chunk.page_content)
    print()

ModuleNotFoundError: No module named 'langchain.text_splitter'

---
## 3. Embeddings

We convert each chunk into a dense vector using `SentenceTransformer` (`all-MiniLM-L6-v2` — fast, 384-dim).

In [ ]:
import numpy as np
import uuid

from sentence_transformers import SentenceTransformer
import chromadb

from typing import List, Any

In [ ]:
class EmbeddingManager:
    """Handles document embeddings using SentenceTransformer"""

    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """
        Initialize the embedding manager.

        Args:
            model_name: HuggingFace model name for sentence embeddings
        """
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Load the SentenceTransformer model"""
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print("Model loaded successfully")
        except Exception as e:
            print(f"Error loading model: {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Generate embeddings for a list of text strings.

        Args:
            texts: List of strings to embed

        Returns:
            numpy array of shape (len(texts), embedding_dim)
        """
        if not texts:
            raise ValueError("texts list cannot be empty")

        print(f"Generating embeddings for {len(texts)} texts...")

        embeddings = self.model.encode(
            texts,
            batch_size=32,
            show_progress_bar=True
        )

        print(f"Generated embeddings with shape: {embeddings.shape}")

        return embeddings


# Initialize
embedding_manager = EmbeddingManager()
embedding_manager

In [31]:
# Convert chunk text into embeddings

texts = [chunk.page_content for chunk in chunks]

embeddings = embedding_manager.generate_embeddings(texts)

print("Embedding shape:", embeddings.shape)

Generating embeddings for 6 texts...


Batches: 100%|██████████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:01<00:00,  1.24s/it]

Generated embeddings with shape: (6, 384)
Embedding shape: (6, 384)


---
## 4. Vector Store (ChromaDB)

We persist the embeddings in a local ChromaDB collection so they survive across sessions.

In [32]:
import os
import uuid
import numpy as np
import chromadb

from typing import List, Any


class VectorStore:
    def __init__(
        self,
        collection_name: str = "pdf_documents",
        persist_directory: str = "data/vector_store"
    ):
        """
        Initialize the vector store.
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None

        self._initialize_store()

    def _initialize_store(self):
        """Initialize ChromaDB client and collection"""
        try:
            os.makedirs(self.persist_directory, exist_ok=True)

            self.client = chromadb.PersistentClient(
                path=self.persist_directory
            )

            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "RAG document embeddings"}
            )

            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents: {self.collection.count()}")

        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Add documents and their embeddings to the vector DB.
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")

        print(f"Adding {len(documents)} documents to vector store...")

        ids, metadatas, documents_text, embeddings_list = [], [], [], []

        for i, (doc, emb) in enumerate(zip(documents, embeddings)):
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)

            metadata = dict(doc.metadata) if hasattr(doc, "metadata") else {}
            metadata["doc_index"] = i
            metadata["content_length"] = len(doc.page_content)

            metadatas.append(metadata)
            documents_text.append(doc.page_content)
            embeddings_list.append(emb.tolist())

        try:
            self.collection.add(
                ids=ids,
                documents=documents_text,
                metadatas=metadatas,
                embeddings=embeddings_list
            )

            print(f"Successfully added {len(documents)} documents")
            print(f"Total documents in collection: {self.collection.count()}")

        except Exception as e:
            print(f"Error adding documents: {e}")
            raise

    def query(
        self,
        query_embedding: np.ndarray,
        n_results: int = 3
    ) -> dict:
        """
        Query the vector store with an embedding vector.

        Args:
            query_embedding: 1-D numpy array (embedding of the query text)
            n_results: number of top results to return

        Returns:
            ChromaDB query result dict
        """
        results = self.collection.query(
            query_embeddings=[query_embedding.tolist()],
            n_results=min(n_results, self.collection.count())
        )
        return results


# Initialize
vector_store = VectorStore()
vector_store

Vector store initialized. Collection: pdf_documents
Existing documents: 6


In [33]:
# Add all chunks + embeddings to ChromaDB
vector_store.add_documents(chunks, embeddings)

Adding 6 documents to vector store...
Successfully added 6 documents
Total documents in collection: 12


---
## 5. RAG Retriever

The retriever embeds a user query and fetches the top-k most similar chunks from ChromaDB.

In [34]:
class RAGRetriever:
    """
    Retrieves relevant document chunks for a given query
    using semantic similarity search over the vector store.
    """

    def __init__(
        self,
        embedding_manager: EmbeddingManager,
        vector_store: VectorStore,
        top_k: int = 3
    ):
        """
        Args:
            embedding_manager : encodes queries into vectors
            vector_store      : ChromaDB wrapper to search against
            top_k             : number of chunks to retrieve
        """
        self.embedding_manager = embedding_manager
        self.vector_store = vector_store
        self.top_k = top_k

    def retrieve(self, query: str) -> List[dict]:
        """
        Retrieve the top-k relevant chunks for a query.

        Args:
            query: natural-language question

        Returns:
            List of dicts with keys: 'content', 'metadata', 'distance'
        """
        print(f"\nQuery: {query!r}")

        # 1. Embed the query
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]

        # 2. Search the vector store
        results = self.vector_store.query(query_embedding, n_results=self.top_k)

        # 3. Format results
        retrieved = []
        documents_list = results.get("documents", [[]])[0]
        metadatas_list = results.get("metadatas", [[]])[0]
        distances_list = results.get("distances", [[]])[0]

        for doc_text, meta, dist in zip(documents_list, metadatas_list, distances_list):
            retrieved.append({
                "content" : doc_text,
                "metadata": meta,
                "distance": dist       # lower = more similar in ChromaDB L2 space
            })

        return retrieved

    def format_context(self, retrieved_chunks: List[dict]) -> str:
        """
        Concatenate retrieved chunks into a single context string
        ready to be injected into a prompt.
        """
        context_parts = []
        for i, chunk in enumerate(retrieved_chunks, 1):
            source = chunk["metadata"].get("source", "unknown")
            context_parts.append(
                f"[Chunk {i} | Source: {source} | Distance: {chunk['distance']:.4f}]\n"
                f"{chunk['content']}"
            )
        return "\n\n".join(context_parts)


# Initialize the retriever
retriever = RAGRetriever(
    embedding_manager=embedding_manager,
    vector_store=vector_store,
    top_k=3
)

print("RAGRetriever initialized successfully")

RAGRetriever initialized successfully


In [35]:
# ----- Test the retriever -----

test_query = "What is RAG and how does it work?"

retrieved_chunks = retriever.retrieve(test_query)

print(f"\nTop {len(retrieved_chunks)} retrieved chunks:\n")

for i, chunk in enumerate(retrieved_chunks, 1):
    print(f"--- Chunk {i} ---")
    print(f"Source  : {chunk['metadata'].get('source', 'N/A')}")
    print(f"Distance: {chunk['distance']:.4f}")
    print(f"Content : {chunk['content']}")
    print()


Query: 'What is RAG and how does it work?'
Generating embeddings for 1 texts...


Batches: 100%|██████████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00, 22.79it/s]

Generated embeddings with shape: (1, 384)

Top 3 retrieved chunks:

--- Chunk 1 ---
Source  : data/text_files/AI_intro.txt
Distance: 1.5209
Content : Retrieval-Augmented Generation (RAG) combines retrieval with large language models for better responses.

--- Chunk 2 ---
Source  : data/text_files/AI_intro.txt
Distance: 1.5209
Content : Retrieval-Augmented Generation (RAG) combines retrieval with large language models for better responses.

--- Chunk 3 ---
Source  : data/text_files/AI_intro.txt
Distance: 1.8164
Content : Natural Language Processing (NLP) focuses on helping computers understand and generate human language.



---
## 6. End-to-End RAG QA (without external LLM)

A lightweight QA function that:
1. Retrieves relevant chunks
2. Builds a prompt with the context
3. Returns the context so you can plug in any LLM (OpenAI, HuggingFace, Ollama, etc.)

> **Tip:** swap `answer_with_context()` with any LLM call to make this fully generative.

In [36]:
def build_rag_prompt(query: str, context: str) -> str:
    """
    Build the final prompt to send to an LLM.

    Args:
        query  : user's question
        context: concatenated retrieved chunks

    Returns:
        Formatted prompt string
    """
    return f"""You are a helpful assistant. Answer the question using ONLY the context below.
If the answer is not in the context, say "I don't have enough information."

Context:
{context}

Question: {query}

Answer:"""


def rag_pipeline(query: str, retriever: RAGRetriever, verbose: bool = True) -> str:
    """
    Full RAG pipeline: retrieve → build prompt → (LLM call placeholder).

    Args:
        query    : user's question
        retriever: initialized RAGRetriever
        verbose  : print intermediate steps

    Returns:
        The prompt ready for an LLM, plus the retrieved context.
    """
    # Step 1: retrieve
    chunks_retrieved = retriever.retrieve(query)

    # Step 2: format context
    context = retriever.format_context(chunks_retrieved)

    # Step 3: build prompt
    prompt = build_rag_prompt(query, context)

    if verbose:
        print("=" * 60)
        print("RETRIEVED CONTEXT")
        print("=" * 60)
        print(context)
        print()
        print("=" * 60)
        print("FINAL PROMPT (send this to your LLM)")
        print("=" * 60)
        print(prompt)

    return prompt, context

In [37]:
# Run the full pipeline

queries = [
    "What is Retrieval-Augmented Generation?",
    "What Python libraries are popular for data science?",
    "How does Machine Learning differ from traditional programming?"
]

for q in queries:
    print(f"\n{'#'*70}")
    print(f"QUESTION: {q}")
    print(f"{'#'*70}")
    prompt, context = rag_pipeline(q, retriever, verbose=True)
    print()


######################################################################
QUESTION: What is Retrieval-Augmented Generation?
######################################################################

Query: 'What is Retrieval-Augmented Generation?'
Generating embeddings for 1 texts...


Batches: 100%|██████████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00, 36.05it/s]


Generated embeddings with shape: (1, 384)
RETRIEVED CONTEXT
[Chunk 1 | Source: data/text_files/AI_intro.txt | Distance: 0.6540]
Retrieval-Augmented Generation (RAG) combines retrieval with large language models for better responses.

[Chunk 2 | Source: data/text_files/AI_intro.txt | Distance: 0.6540]
Retrieval-Augmented Generation (RAG) combines retrieval with large language models for better responses.

[Chunk 3 | Source: data/text_files/AI_intro.txt | Distance: 1.5271]
Natural Language Processing (NLP) focuses on helping computers understand and generate human language.

FINAL PROMPT (send this to your LLM)
You are a helpful assistant. Answer the question using ONLY the context below.
If the answer is not in the context, say "I don't have enough information."

Context:
[Chunk 1 | Source: data/text_files/AI_intro.txt | Distance: 0.6540]
Retrieval-Augmented Generation (RAG) combines retrieval with large language models for better responses.

[Chunk 2 | Source: data/text_files/AI_intro.

Batches: 100%|██████████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00, 97.50it/s]


Generated embeddings with shape: (1, 384)
RETRIEVED CONTEXT
[Chunk 1 | Source: data/text_files/python_intro.txt | Distance: 0.6763]
Python has simple syntax, making it easy to learn and use.

Popular Python libraries include NumPy, Pandas, TensorFlow, and Flask.

[Chunk 2 | Source: data/text_files/python_intro.txt | Distance: 0.6763]
Python has simple syntax, making it easy to learn and use.

Popular Python libraries include NumPy, Pandas, TensorFlow, and Flask.

[Chunk 3 | Source: data/text_files/python_intro.txt | Distance: 0.7706]
Python is a powerful and beginner-friendly programming language.

It is widely used in web development, data science, machine learning, automation, and artificial intelligence.

FINAL PROMPT (send this to your LLM)
You are a helpful assistant. Answer the question using ONLY the context below.
If the answer is not in the context, say "I don't have enough information."

Context:
[Chunk 1 | Source: data/text_files/python_intro.txt | Distance: 0.6763]
Python h

Batches: 100%|█████████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00, 105.80it/s]

Generated embeddings with shape: (1, 384)
RETRIEVED CONTEXT
[Chunk 1 | Source: data/text_files/AI_intro.txt | Distance: 0.7606]
Machine Learning (ML), a subset of AI, allows systems to learn patterns from data and improve over time without being explicitly programmed.

[Chunk 2 | Source: data/text_files/AI_intro.txt | Distance: 0.7606]
Machine Learning (ML), a subset of AI, allows systems to learn patterns from data and improve over time without being explicitly programmed.

[Chunk 3 | Source: data/text_files/python_intro.txt | Distance: 1.1710]
Python has simple syntax, making it easy to learn and use.

Popular Python libraries include NumPy, Pandas, TensorFlow, and Flask.

FINAL PROMPT (send this to your LLM)
You are a helpful assistant. Answer the question using ONLY the context below.
If the answer is not in the context, say "I don't have enough information."

Context:
[Chunk 1 | Source: data/text_files/AI_intro.txt | Distance: 0.7606]
Machine Learning (ML), a subset of AI, allows 

---
## 7. PDF Ingestion

Two sub-sections:
- **7a** — Download a demo PDF automatically (no files needed)
- **7b** — Load your own PDFs from `data/pdf/`

Both use `PyMuPDFLoader` which preserves page numbers in metadata.

In [38]:
# Install PDF parser if not already installed
# !pip install pymupdf langchain-community

### 7a. Demo PDF — auto-downloaded sample

In [39]:
import os
import urllib.request

os.makedirs("data/pdf", exist_ok=True)

# Publicly available ML cheat-sheet PDF (small, ~200 KB)
DEMO_PDF_URL  = "https://raw.githubusercontent.com/afshinea/stanford-cs-229-machine-learning/master/en/super-cheatsheet-machine-learning.pdf"
DEMO_PDF_PATH = "data/pdf/ml_cheatsheet.pdf"

if not os.path.exists(DEMO_PDF_PATH):
    print("Downloading demo PDF...")
    urllib.request.urlretrieve(DEMO_PDF_URL, DEMO_PDF_PATH)
    print(f"Saved to: {DEMO_PDF_PATH}")
else:
    print(f"Demo PDF already exists: {DEMO_PDF_PATH}")

Saved to: data/pdf/ml_cheatsheet.pdf


In [40]:
from langchain_community.document_loaders import PyMuPDFLoader

# Load the demo PDF — each page becomes one Document
pdf_loader = PyMuPDFLoader(DEMO_PDF_PATH)
pdf_docs   = pdf_loader.load()

print(f"Pages loaded : {len(pdf_docs)}")
print(f"\n--- Page 1 metadata ---")
print(pdf_docs[0].metadata)
print(f"\n--- Page 1 content (first 300 chars) ---")
print(pdf_docs[0].page_content[:300])

Pages loaded : 16

--- Page 1 metadata ---
{'producer': 'pdfTeX-1.40.19', 'creator': 'LaTeX with hyperref package', 'creationdate': '2018-10-06T20:03:13-07:00', 'source': 'data/pdf/ml_cheatsheet.pdf', 'file_path': 'data/pdf/ml_cheatsheet.pdf', 'total_pages': 16, 'format': 'PDF 1.5', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': '2018-10-06T20:03:13-07:00', 'trapped': '', 'modDate': "D:20181006200313-07'00'", 'creationDate': "D:20181006200313-07'00'", 'page': 0}

--- Page 1 content (first 300 chars) ---
CS 229 – Machine Learning
https://stanford.edu/~shervine
Super VIP Cheatsheet: Machine Learning
Afshine Amidi and Shervine Amidi
October 6, 2018
Contents
1
Supervised Learning
2
1.1
Introduction to Supervised Learning . . . . . . . . . . . . . . . . . . .
2
1.2
Notations and general concepts
. . . .


### 7b. Your own PDFs — load an entire folder

In [41]:
from langchain_community.document_loaders import DirectoryLoader, PyMuPDFLoader

# Drop any .pdf files into data/pdf/ and they'll be picked up here
dir_pdf = DirectoryLoader(
    "data/pdf/",
    glob="**/*.pdf",
    loader_cls=PyMuPDFLoader,
    show_progress=True
)

all_pdf_docs = dir_pdf.load()

print(f"Total PDF pages loaded: {len(all_pdf_docs)}")

# Show sources found
sources = set(d.metadata.get("source", "unknown") for d in all_pdf_docs)
print(f"PDF files processed  : {len(sources)}")
for s in sources:
    print(f"  - {s}")

100%|███████████████████████████████████████████████████████████████████████████████████████████████████| 4/4 [00:00<00:00,  7.49it/s]

Total PDF pages loaded: 69
PDF files processed  : 4
  - data/pdf/Smart pH-Responsive polymers in biomedical Applications.pdf
  - data/pdf/ml_cheatsheet.pdf
  - data/pdf/SESV4I2202638.pdf
  - data/pdf/The future of pharmaceuticals-.pdf


In [43]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

pdf_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,    # PDFs often have denser text — larger chunks work better
    chunk_overlap=80,
    separators=["\n\n", "\n", " ", ""]
)

pdf_chunks = pdf_splitter.split_documents(all_pdf_docs)

print(f"PDF pages  : {len(all_pdf_docs)}")
print(f"PDF chunks : {len(pdf_chunks)}")
print()
print("Sample chunk:")
print(pdf_chunks[0].page_content[:300])

PDF pages  : 69
PDF chunks : 957

Sample chunk:
CS 229 – Machine Learning
https://stanford.edu/~shervine
Super VIP Cheatsheet: Machine Learning
Afshine Amidi and Shervine Amidi
October 6, 2018
Contents
1
Supervised Learning
2
1.1
Introduction to Supervised Learning . . . . . . . . . . . . . . . . . . .
2
1.2
Notations and general concepts
. . . .


In [44]:
# Embed PDF chunks and store them in a separate ChromaDB collection

pdf_texts      = [chunk.page_content for chunk in pdf_chunks]
pdf_embeddings = embedding_manager.generate_embeddings(pdf_texts)

# Use a dedicated collection so PDF data is separate from text data
pdf_vector_store = VectorStore(
    collection_name="pdf_rag_collection",
    persist_directory="data/vector_store_pdf"
)

pdf_vector_store.add_documents(pdf_chunks, pdf_embeddings)

# Build a retriever for PDF content
pdf_retriever = RAGRetriever(
    embedding_manager=embedding_manager,
    vector_store=pdf_vector_store,
    top_k=3
)

print("PDF retriever ready!")

Generating embeddings for 957 texts...


Batches: 100%|████████████████████████████████████████████████████████████████████████████████████████| 30/30 [00:08<00:00,  3.35it/s]


Generated embeddings with shape: (957, 384)
Vector store initialized. Collection: pdf_rag_collection
Existing documents: 0
Adding 957 documents to vector store...
Successfully added 957 documents
Total documents in collection: 957
PDF retriever ready!


In [45]:
# Quick retrieval test on PDF content
pdf_query   = "What is regularization in machine learning?"
pdf_results = pdf_retriever.retrieve(pdf_query)

print(f"\nQuery: {pdf_query!r}\n")
for i, r in enumerate(pdf_results, 1):
    print(f"--- Result {i} | Source: {r['metadata'].get('source','?')} | Page: {r['metadata'].get('page','?')} ---")
    print(r["content"][:300])
    print()


Query: 'What is regularization in machine learning?'
Generating embeddings for 1 texts...


Batches: 100%|██████████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00,  1.95it/s]

Generated embeddings with shape: (1, 384)

Query: 'What is regularization in machine learning?'

--- Result 1 | Source: data/pdf/ml_cheatsheet.pdf | Page: 10 ---
- Generally k = 5 or 10
- Case p = 1 is called leave-one-out
The most commonly used method is called k-fold cross-validation and splits the training data
into k folds to validate the model on one fold while training the model on the k −1 other folds,
all of this k times. The error is then averaged o

--- Result 2 | Source: data/pdf/The future of pharmaceuticals-.pdf | Page: 2 ---
reacting to it to achieve speciﬁc objectives (Fig. 2). The fundamental
approach involves “training” machines with algorithms and data to
enable them to perform tasks and make predictions or inferences
about future outcomes. The industry often classiﬁes ML algorithms
in two ways: based on learning sc

--- Result 3 | Source: data/pdf/ml_cheatsheet.pdf | Page: 10 ---
data and thus deals with high variance issues. The following table sums up the diﬀerent 

---
## 8. HuggingFace LLM Integration (Free — Mistral / Zephyr)

Uses the **HuggingFace Inference API** — completely free with a HF account.

### Setup
1. Go to [huggingface.co/settings/tokens](https://huggingface.co/settings/tokens)
2. Create a **Read** token (free)
3. Paste it below

### Model choices
| Model | HF ID | Notes |
|-------|-------|-------|
| **Mistral 7B Instruct** | `mistralai/Mistral-7B-Instruct-v0.3` | Best quality, default |
| **Zephyr 7B Beta** | `HuggingFaceH4/zephyr-7b-beta` | Good instruction following |
| **Phi-3 Mini** | `microsoft/Phi-3-mini-4k-instruct` | Smallest, fastest |

In [46]:
# !pip install huggingface_hub

In [47]:
import os
from huggingface_hub import InferenceClient

# -------------------------------------------------------------------
# Paste your HuggingFace token here  (or set env var HF_TOKEN)
# Get one free at: https://huggingface.co/settings/tokens
# -------------------------------------------------------------------
HF_TOKEN = os.environ.get("HF_TOKEN", "hf_YOUR_TOKEN_HERE")

# Choose your model
HF_MODEL = "mistralai/Mistral-7B-Instruct-v0.3"   # swap to zephyr-7b-beta if preferred

hf_client = InferenceClient(
    model=HF_MODEL,
    token=HF_TOKEN
)

print(f"HuggingFace client ready — model: {HF_MODEL}")

HuggingFace client ready — model: mistralai/Mistral-7B-Instruct-v0.3


In [48]:
class HuggingFaceRAG:
    """
    Wraps the HuggingFace Inference API to answer questions
    using retrieved RAG context.
    """

    def __init__(
        self,
        hf_client: InferenceClient,
        retriever: RAGRetriever,
        max_new_tokens: int = 512,
        temperature: float = 0.3
    ):
        self.client         = hf_client
        self.retriever      = retriever
        self.max_new_tokens = max_new_tokens
        self.temperature    = temperature

    def _build_prompt(self, query: str, context: str) -> str:
        """Mistral/Zephyr instruction format: [INST] ... [/INST]"""
        return (
            "[INST] You are a helpful assistant. "
            "Answer the question using ONLY the context provided. "
            "If the answer is not in the context, say 'I don't have enough information.'\n\n"
            f"Context:\n{context}\n\n"
            f"Question: {query} [/INST]"
        )

    def ask(self, query: str, verbose: bool = True) -> str:
        """
        Full RAG + LLM pipeline:
          1. Retrieve relevant chunks
          2. Build prompt
          3. Call HuggingFace Inference API
          4. Return the answer

        Args:
            query  : user question
            verbose: print retrieved context

        Returns:
            LLM answer string
        """
        # Step 1 — Retrieve
        chunks_retrieved = self.retriever.retrieve(query)
        context          = self.retriever.format_context(chunks_retrieved)

        if verbose:
            print("\n" + "=" * 60)
            print("RETRIEVED CONTEXT")
            print("=" * 60)
            print(context)

        # Step 2 — Build prompt
        prompt = self._build_prompt(query, context)

        # Step 3 — Call HuggingFace Inference API
        try:
            response = self.client.text_generation(
                prompt,
                max_new_tokens=self.max_new_tokens,
                temperature=self.temperature,
                do_sample=True,
                return_full_text=False   # only return the generated part
            )
            answer = response.strip()
        except Exception as e:
            answer = f"[LLM Error] {e}"

        if verbose:
            print("\n" + "=" * 60)
            print("LLM ANSWER")
            print("=" * 60)
            print(answer)

        return answer


# Initialize with the text retriever (swap pdf_retriever for PDF QA)
rag_llm = HuggingFaceRAG(
    hf_client=hf_client,
    retriever=retriever,        # use pdf_retriever to query PDFs instead
    max_new_tokens=512,
    temperature=0.3
)

print("HuggingFaceRAG pipeline ready!")

HuggingFaceRAG pipeline ready!


In [49]:
# ---- Ask a question (text documents) ----

answer = rag_llm.ask(
    "What is Retrieval-Augmented Generation and why is it useful?",
    verbose=True
)


Query: 'What is Retrieval-Augmented Generation and why is it useful?'
Generating embeddings for 1 texts...


Batches: 100%|██████████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00,  2.70it/s]


Generated embeddings with shape: (1, 384)

RETRIEVED CONTEXT
[Chunk 1 | Source: data/text_files/AI_intro.txt | Distance: 0.6206]
Retrieval-Augmented Generation (RAG) combines retrieval with large language models for better responses.

[Chunk 2 | Source: data/text_files/AI_intro.txt | Distance: 0.6206]
Retrieval-Augmented Generation (RAG) combines retrieval with large language models for better responses.

[Chunk 3 | Source: data/text_files/AI_intro.txt | Distance: 1.4944]
Natural Language Processing (NLP) focuses on helping computers understand and generate human language.

LLM ANSWER
[LLM Error] Model mistralai/Mistral-7B-Instruct-v0.3 is not supported for task text-generation and provider novita. Supported task: conversational.


In [50]:
# ---- Ask a question over PDF content ----

pdf_rag_llm = HuggingFaceRAG(
    hf_client=hf_client,
    retriever=pdf_retriever,   # PDF-backed retriever
    max_new_tokens=512,
    temperature=0.3
)

answer_pdf = pdf_rag_llm.ask(
    "Explain regularization and when to use L1 vs L2.",
    verbose=True
)


Query: 'Explain regularization and when to use L1 vs L2.'
Generating embeddings for 1 texts...


Batches: 100%|██████████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00,  1.67it/s]

Generated embeddings with shape: (1, 384)

RETRIEVED CONTEXT
[Chunk 1 | Source: data/pdf/ml_cheatsheet.pdf | Distance: 1.0762]
data and thus deals with high variance issues. The following table sums up the diﬀerent types
of commonly used regularization techniques:
LASSO
Ridge
Elastic Net
- Shrinks coeﬃcients to 0
Makes coeﬃcients smaller
Tradeoﬀbetween variable
- Good for variable selection
selection and small coeﬃcients
... + λ||θ||1
... + λ||θ||2
2
... + λ
h
(1 −α)||θ||1 + α||θ||2
2
i
λ ∈R
λ ∈R
λ ∈R,
α ∈[0,1]
Ì Model selection – Train model on training set, then evaluate on the development set, then

[Chunk 2 | Source: data/pdf/ml_cheatsheet.pdf | Distance: 1.1665]
- Generally k = 5 or 10
- Case p = 1 is called leave-one-out
The most commonly used method is called k-fold cross-validation and splits the training data
into k folds to validate the model on one fold while training the model on the k −1 other folds,
all of this k times. The error is then averaged over the k folds and is n

In [51]:
# ---- Multi-turn: ask several questions in a loop ----

questions = [
    "What is NLP?",
    "What Python libraries are commonly used in data science?",
    "How does machine learning work?"
]

for q in questions:
    print(f"\n{'#'*70}")
    print(f"Q: {q}")
    print(f"{'#'*70}")
    ans = rag_llm.ask(q, verbose=False)   # verbose=False → only print answer
    print(f"A: {ans}")


######################################################################
Q: What is NLP?
######################################################################

Query: 'What is NLP?'
Generating embeddings for 1 texts...


Batches: 100%|██████████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00,  2.23it/s]


Generated embeddings with shape: (1, 384)
A: [LLM Error] Model mistralai/Mistral-7B-Instruct-v0.3 is not supported for task text-generation and provider novita. Supported task: conversational.

######################################################################
Q: What Python libraries are commonly used in data science?
######################################################################

Query: 'What Python libraries are commonly used in data science?'
Generating embeddings for 1 texts...


Batches: 100%|██████████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00,  3.05it/s]


Generated embeddings with shape: (1, 384)
A: [LLM Error] Model mistralai/Mistral-7B-Instruct-v0.3 is not supported for task text-generation and provider novita. Supported task: conversational.

######################################################################
Q: How does machine learning work?
######################################################################

Query: 'How does machine learning work?'
Generating embeddings for 1 texts...


Batches: 100%|██████████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00,  2.43it/s]

Generated embeddings with shape: (1, 384)
A: [LLM Error] Model mistralai/Mistral-7B-Instruct-v0.3 is not supported for task text-generation and provider novita. Supported task: conversational.


---
## Next Steps

| Step | What to do |
|------|------------|
| **Switch model** | Change `HF_MODEL` to `HuggingFaceH4/zephyr-7b-beta` or `microsoft/Phi-3-mini-4k-instruct` |
| **Add your PDFs** | Drop `.pdf` files into `data/pdf/` and re-run Section 7b |
| **Better chunking** | Try `SemanticChunker` from `langchain_experimental` for semantic-aware splits |
| **Reranking** | Add a cross-encoder (`cross-encoder/ms-marco-MiniLM-L-6-v2`) to rerank retrieved chunks |
| **LangChain LCEL** | Wire everything with `RunnablePassthrough` + `HuggingFaceEndpoint` for a clean chain |
| **Streaming** | Pass `stream=True` to `hf_client.text_generation()` for token-by-token output |